# Prophet robustness: dedicated sweep (expected-zero-variance check)

Prophet is fully deterministic given identical input -- no seed anywhere in its fit.
Run twice per dataset anyway (`seed_43`/`seed_44`) for structural consistency with
the other robustness notebooks; expect std dev = 0.

Calls `prophet_baseline.py::run_prophet_baseline()` with `output_dir` redirected to
`robustness/prophet/<sub>/<dataset>/seed_<N>/`. Fully resumable -- skips a
`(dataset, seed)` combination if its `metrics.csv` already exists.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))

from prophet_baseline import run_prophet_baseline

RESULTS = ROOT / "results"
ROBUST = ROOT / "robustness" / "prophet"

SYNTH_DATASETS = [p.stem for p in sorted((ROOT / "data" / "synthetic").glob("*.xes")) if "recency" not in p.stem]
REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                 "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]
SERIES = ["concurrent_cases", "throughput_time"]

NEW_SEEDS = [43, 44]

print(f"{len(SYNTH_DATASETS)} synthetic datasets, {len(REAL_DATASETS)} real-life (ssd) datasets, seeds={NEW_SEEDS}")


In [ ]:
def run_prophet_robustness_one(dataset: str, is_real: bool, seed: int):
    """Reshapes prophet_baseline.py's own output into the standard robustness schema."""
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST / sub / dataset / f"seed_{seed}"
    metrics_path = out_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}: metrics already exist")
        return

    trim = "ssd" if is_real else "none"
    xes_path = (ROOT / "data" / "real-life" / f"{dataset}.xes") if is_real \
        else (ROOT / "data" / "synthetic" / f"{dataset}.xes")

    out_dir.mkdir(parents=True, exist_ok=True)
    result = run_prophet_baseline(dataset, xes_path, trim=trim, is_real=is_real,
                                  output_dir=out_dir, overwrite=True)
    if result.empty:
        print(f"  [skip] {dataset}/seed_{seed}: no rows produced")
        return

    rows = result[["dataset", "series", "mae", "mse"]].copy()
    rows["model"] = "prophet"
    rows["seed"] = seed
    rows = rows[["dataset", "series", "model", "seed", "mae", "mse"]]
    rows.to_csv(metrics_path, index=False)
    print(f"  [done] {dataset}/seed_{seed}: {len(rows)} (model, series) rows")


## Part 1: Synthetic

In [ ]:
for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (synthetic)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_prophet_robustness_one(name, is_real=False, seed=seed)


## Part 2: SSD

In [ ]:
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (ssd)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_prophet_robustness_one(name, is_real=True, seed=seed)
